# Practical 6: Balance the Dataset for ML/DL Training

**Objective:** Handle imbalanced dataset for fair ML training.

- Analyze distribution of benign vs. attack traffic
- Use undersampling of benign / oversampling of attack (SMOTE or random oversampling)
- Prepare a balanced dataset

In [7]:
import pandas as pd

df = pd.read_csv('logs/features_access_log.csv')
print('Rows loaded:', len(df))
df['label'].value_counts()

Rows loaded: 47685


label
benign            43233
brute_force        2485
path_traversal     1000
sqli                967
Name: count, dtype: int64

## 1. Visualize the imbalance

`benign` dwarfs every attack class — expected for real traffic, but a classifier trained directly on this would just learn to always predict `benign` and still score >90% accuracy while catching zero attacks. That's exactly the failure mode balancing fixes.

In [8]:
counts = df['label'].value_counts()
percentages = (counts / counts.sum() * 100).round(2)
pd.DataFrame({'count': counts, 'percent': percentages})

,count,percent
label,,
benign,43233,90.66
brute_force,2485,5.21
path_traversal,1000,2.10
sqli,967,2.03


## 2. Prepare features for resampling

SMOTE and the samplers need purely numeric input, so categorical columns (`Request Type`, `ua_type`) get one-hot encoded first. Identifier/free-text columns (`IP Address`, `Date/Time`, `Resource`) are dropped here — they're not model features, just row-level metadata.

In [9]:
model_df = df.drop(columns=['IP Address', 'Date/Time', 'Resource'])
model_df = pd.get_dummies(model_df, columns=['Request Type', 'ua_type'], drop_first=False)

X = model_df.drop(columns=['label'])
y = model_df['label']
X.head()

,Status Code,requests_per_ip,time_since_prev_request,is_error_status,error_rate_per_ip,url_entropy,Request Type_GET,Request Type_POST,ua_type_bot,ua_type_browser
0,200,1,9999.0,0,0.0,3.169925,True,False,False,True
1,200,1,9999.0,0,0.0,2.321928,True,False,False,True
2,200,1,9999.0,0,0.0,2.584963,True,False,False,True
3,500,1,9999.0,1,1.0,3.773557,True,False,False,True
4,200,1,9999.0,0,0.0,3.750000,True,False,False,True


## 3. Combined approach: undersample benign, oversample attacks (SMOTE)

Going straight from ~40,000 benign to matching the smallest class with pure SMOTE would synthesize an enormous number of fake minority rows. Instead: first undersample `benign` down to a reasonable ceiling, then SMOTE the attack classes up to match it. This keeps the dataset size sane while still balancing the classes.

In [10]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Step 1: cap benign at 3000 (still comfortably the plurality, but not overwhelming)
undersample = RandomUnderSampler(
    sampling_strategy={'benign': 3000},
    random_state=42
)

# Step 2: SMOTE every attack class up to 3000 as well
oversample = SMOTE(
    sampling_strategy={'sqli': 3000, 'path_traversal': 3000, 'brute_force': 3000},
    random_state=42,
    k_neighbors=5
)

pipeline = ImbPipeline(steps=[('under', undersample), ('over', oversample)])
X_balanced, y_balanced = pipeline.fit_resample(X, y)

print('Before:', dict(y.value_counts()))
print('After: ', dict(y_balanced.value_counts()))

Before: {'benign': np.int64(43233), 'brute_force': np.int64(2485), 'path_traversal': np.int64(1000), 'sqli': np.int64(967)}
After:  {'benign': np.int64(3000), 'brute_force': np.int64(3000), 'path_traversal': np.int64(3000), 'sqli': np.int64(3000)}


## Observations

- All four classes now sit at 3000 rows each, a balanced 25/25/25/25 split.
- SMOTE does not duplicate real rows for the minority classes; it interpolates *synthetic* points between existing minority neighbors in feature space. This requires at least `k_neighbors` real examples per class, which was satisfied here since even the smallest class (`brute_force`) had a few thousand rows.
- Undersampling `benign` discards real data points, a trade-off against the alternative of pure oversampling (which would instead synthesize a very large number of minority rows to match the majority class size).

## Save the balanced dataset for Practical 7

In [11]:
balanced_df = X_balanced.copy()
balanced_df['label'] = y_balanced
balanced_df.to_csv('logs/balanced_access_log.csv', index=False)
print('Saved logs/balanced_access_log.csv')

Saved logs/balanced_access_log.csv
